In [1]:
import pandas as pd
import os

# Test that we can see the data folders
ae_path = '../data/raw/ae'
national_path = '../data/raw/sickness/national'
trust_path = '../data/raw/sickness/trust'

print("A&E files:", len(os.listdir(ae_path)))
print("National sickness files:", len(os.listdir(national_path)))
print("Trust sickness files:", len(os.listdir(trust_path)))

A&E files: 48
National sickness files: 21
Trust sickness files: 25


In [2]:
# Load one A&E file to inspect the structure
sample_ae = pd.read_csv('../data/raw/ae/' + os.listdir(ae_path)[0])
print("Columns:", list(sample_ae.columns))
print("Shape:", sample_ae.shape)
sample_ae.head(3)

Columns: ['Period', 'Org Code', 'Parent Org', 'Org name', 'A&E attendances Type 1', 'A&E attendances Type 2', 'A&E attendances Other A&E Department', 'A&E attendances Booked Appointments Type 1', 'A&E attendances Booked Appointments Type 2', 'A&E attendances Booked Appointments Other Department', 'Attendances over 4hrs Type 1', 'Attendances over 4hrs Type 2', 'Attendances over 4hrs Other Department', 'Attendances over 4hrs Booked Appointments Type 1', 'Attendances over 4hrs Booked Appointments Type 2', 'Attendances over 4hrs Booked Appointments Other Department', 'Patients who have waited 4-12 hs from DTA to admission', 'Patients who have waited 12+ hrs from DTA to admission', 'Emergency admissions via A&E - Type 1', 'Emergency admissions via A&E - Type 2', 'Emergency admissions via A&E - Other A&E department', 'Other emergency admissions']
Shape: (205, 22)


,Period,Org Code,Parent Org,Org name,A&E attendances Type 1,A&E attendances Type 2,A&E attendances Other A&E Department,A&E attendances Booked Appointments Type 1,A&E attendances Booked Appointments Type 2,A&E attendances Booked Appointments Other Department,...,Attendances over 4hrs Other Department,Attendances over 4hrs Booked Appointments Type 1,Attendances over 4hrs Booked Appointments Type 2,Attendances over 4hrs Booked Appointments Other Department,Patients who have waited 4-12 hs from DTA to admission,Patients who have waited 12+ hrs from DTA to admission,Emergency admissions via A&E - Type 1,Emergency admissions via A&E - Type 2,Emergency admissions via A&E - Other A&E department,Other emergency admissions
0,MSitAE-APRIL-2022,RLQ,NHS ENGLAND MIDLANDS,WYE VALLEY NHS TRUST,5536,0,0,139,0,0,...,0,31,0,0,179,270,1257,0,0,388
1,MSitAE-APRIL-2022,AD913,NHS ENGLAND LONDON,BECKENHAM BEACON UCC,0,0,3426,0,0,252,...,219,0,0,28,0,0,0,0,0,0
2,MSitAE-APRIL-2022,AJN,NHS ENGLAND NORTH EAST AND YORKSHIRE,WORKINGTON HEALTH LIMITED,0,0,312,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [3]:
# Load all 48 A&E files and combine
import glob
from dateutil.parser import parse

def parse_ae_period(period_str):
    # Convert MSitAE-APRIL-2022 to a proper date
    parts = period_str.replace('MSitAE-', '').replace('MSitAE -', '')
    return pd.to_datetime(parts, format='%B-%Y')

all_ae_files = glob.glob(ae_path + '/*.csv')
ae_frames = []

for f in all_ae_files:
    df = pd.read_csv(f)
    ae_frames.append(df)

ae_combined = pd.concat(ae_frames, ignore_index=True)
ae_combined['date'] = ae_combined['Period'].str.replace('MSitAE-', '').str.replace('MSitAE -', '').str.strip()
ae_combined['date'] = pd.to_datetime(ae_combined['date'], format='%B-%Y')

print("Total rows:", len(ae_combined))
print("Date range:", ae_combined['date'].min(), "to", ae_combined['date'].max())
print("Unique trusts:", ae_combined['Org Code'].nunique())

ValueError: time data "Total" doesn't match format "%B-%Y". You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [4]:
# Load all 48 A&E files and combine - with summary row fix
all_ae_files = glob.glob(ae_path + '/*.csv')
ae_frames = []

for f in all_ae_files:
    df = pd.read_csv(f)
    ae_frames.append(df)

ae_combined = pd.concat(ae_frames, ignore_index=True)

# Clean the period column
ae_combined['date_str'] = ae_combined['Period'].str.replace('MSitAE-', '').str.replace('MSitAE -', '').str.strip()

# Remove any rows where date_str is not a valid month-year (e.g. 'Total')
ae_combined = ae_combined[~ae_combined['date_str'].str.contains('Total|total', na=True)]

# Parse to datetime
ae_combined['date'] = pd.to_datetime(ae_combined['date_str'], format='%B-%Y')

print("Total rows:", len(ae_combined))
print("Date range:", ae_combined['date'].min(), "to", ae_combined['date'].max())
print("Unique trusts:", ae_combined['Org Code'].nunique())

ValueError: time data "TOTAl" doesn't match format "%B-%Y". You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [5]:
# Load all 48 A&E files - robust version
import re

all_ae_files = glob.glob(ae_path + '/*.csv')
ae_frames = []

for f in all_ae_files:
    df = pd.read_csv(f)
    ae_frames.append(df)

ae_combined = pd.concat(ae_frames, ignore_index=True)

# Clean period column
ae_combined['date_str'] = ae_combined['Period'].str.replace('MSitAE-', '').str.replace('MSitAE -', '').str.strip()

# Only keep rows where date_str matches MONTH-YEAR pattern
valid_months = ['JANUARY','FEBRUARY','MARCH','APRIL','MAY','JUNE',
                'JULY','AUGUST','SEPTEMBER','OCTOBER','NOVEMBER','DECEMBER']
ae_combined = ae_combined[ae_combined['date_str'].str.upper().str.split('-').str[0].isin(valid_months)]

# Parse to datetime
ae_combined['date'] = pd.to_datetime(ae_combined['date_str'], format='%B-%Y')

print("Total rows:", len(ae_combined))
print("Date range:", ae_combined['date'].min(), "to", ae_combined['date'].max())
print("Unique trusts:", ae_combined['Org Code'].nunique())

Total rows: 9625
Date range: 2022-04-01 00:00:00 to 2026-03-01 00:00:00
Unique trusts: 222


In [6]:
# Load all 25 trust-level sickness files
all_trust_files = glob.glob(trust_path + '/*.csv')
trust_frames = []

for f in all_trust_files:
    df = pd.read_csv(f)
    trust_frames.append(df)

trust_combined = pd.concat(trust_frames, ignore_index=True)

# Parse date column
trust_combined['date'] = pd.to_datetime(trust_combined['DATE'], dayfirst=True)

# Keep only all staff groups and all reasons total
trust_combined = trust_combined[
    (trust_combined['STAFF_GROUP'] == 'All staff groups') &
    (trust_combined['REASON'] == 'ALL REASONS')
]

print("Total rows:", len(trust_combined))
print("Date range:", trust_combined['date'].min(), "to", trust_combined['date'].max())
print("Unique trusts:", trust_combined['ORG_CODE'].nunique())

Total rows: 0
Date range: NaT to NaT
Unique trusts: 0


In [7]:
# Check actual values in the trust sickness files
sample_trust = pd.read_csv(all_trust_files[0])
print("Columns:", list(sample_trust.columns))
print()
print("STAFF_GROUP unique values:", sample_trust['STAFF_GROUP'].unique())
print()
print("REASON unique values:", sample_trust['REASON'].unique()[:5])
print()
print("DATE sample:", sample_trust['DATE'].head(3).values)

Columns: ['DATE', 'NHSE_CODE', 'NHSE_NAME', 'ORG_CODE', 'ORG_NAME', 'STAFF_GROUP', 'REASON', 'FTE_DAYS_AVAILABLE', 'FTE_DAYS_LOST', 'FTE_DAYS_LOST_REASON']

STAFF_GROUP unique values: <StringArray>
[                                         'All staff groups',
                                'NHS infrastructure support',
          'Other staff or those with unknown classification',
                   'Professionally qualified clinical staff',
                                 'Support to clinical staff',
                                           'Ambulance staff',
                                         'Central functions',
                                 'HCHS doctors - All grades',
                                 'Hotel, property & estates',
                                                  'Managers',
                                                  'Midwives',
                                  'Nurses & health visitors',
                 'Scientific, therapeutic & technical staf

In [8]:
# Reload and aggregate correctly
all_trust_files = glob.glob(trust_path + '/*.csv')
trust_frames = []

for f in all_trust_files:
    df = pd.read_csv(f)
    trust_frames.append(df)

trust_raw = pd.concat(trust_frames, ignore_index=True)

# Parse date
trust_raw['date'] = pd.to_datetime(trust_raw['DATE'], dayfirst=True)

# Filter to All staff groups only, then sum all reasons per trust per month
trust_combined = trust_raw[trust_raw['STAFF_GROUP'] == 'All staff groups'].groupby(
    ['date', 'ORG_CODE', 'ORG_NAME'], as_index=False
).agg(
    FTE_DAYS_LOST=('FTE_DAYS_LOST', 'sum'),
    FTE_DAYS_AVAILABLE=('FTE_DAYS_AVAILABLE', 'max')
)

# Calculate sickness rate
trust_combined['sickness_rate'] = trust_combined['FTE_DAYS_LOST'] / trust_combined['FTE_DAYS_AVAILABLE'] * 100

print("Total rows:", len(trust_combined))
print("Date range:", trust_combined['date'].min(), "to", trust_combined['date'].max())
print("Unique trusts:", trust_combined['ORG_CODE'].nunique())
trust_combined.head(3)

Total rows: 7067
Date range: 2024-01-31 00:00:00 to 2026-01-31 00:00:00
Unique trusts: 288


,date,ORG_CODE,ORG_NAME,FTE_DAYS_LOST,FTE_DAYS_AVAILABLE,sickness_rate
0,2024-01-31,0AR,NHS North of England Commissioning Support Unit,43977.42573,57267.23704,76.793343
1,2024-01-31,0CX,NHS Midlands and Lancashire Commissioning Supp...,41265.94516,50516.56125,81.687954
2,2024-01-31,0DE,NHS Arden and Greater East Midlands Commission...,17493.08416,35916.18362,48.705298


In [9]:
# Save both cleaned datasets to the processed folder
ae_combined.to_csv('../data/processed/ae_clean.csv', index=False)
trust_combined.to_csv('../data/processed/sickness_trust_clean.csv', index=False)

print("ae_clean.csv saved:", len(ae_combined), "rows")
print("sickness_trust_clean.csv saved:", len(trust_combined), "rows")

ae_clean.csv saved: 9625 rows
sickness_trust_clean.csv saved: 7067 rows
